# Transfer Tax Reform Impact on Housing Production

**Research Question:** How would eliminating San Francisco's real estate transfer tax affect housing production?

**Methodology:** We load estimated market values for all SF residential parcels (from `estimate-parcel-market-value/output.csv`), calculate how removing the transfer tax would reduce effective construction costs (land is ~2x construction costs, so the tax has a 2x impact on effective costs), then run the City Economist's housing projection model with adjusted costs to estimate the unit difference.

In [1]:
import pandas as pd
from pathlib import Path
import subprocess
import json
import tempfile

PROJECT_ROOT = Path.cwd().parent.parent

In [2]:
input_path = Path('../estimate-parcel-market-value/output.csv')
df = pd.read_csv(input_path)
print(f"Loaded {len(df):,} parcels")

Loaded 191,090 parcels


In [3]:
df['mapblklot'] = df['parcel_number'].astype(str).str[:7]

mapblklot_values = df.groupby('mapblklot')['market_value'].sum().reset_index()
mapblklot_values.columns = ['mapblklot', 'total_value']
print(f"Aggregated {len(df):,} parcels into {len(mapblklot_values):,} mapblklots")

Aggregated 191,090 parcels into 162,610 mapblklots


In [4]:
BASE_CONSTRUCTION_COST = 112.723
TAX_TO_COST_MULTIPLIER = 2

def get_transfer_tax_rate(value):
    if value >= 25_000_000: return 0.06
    if value >= 10_000_000: return 0.055
    if value >= 5_000_000: return 0.0225
    if value >= 1_000_000: return 0.0075
    if value >= 250_001: return 0.0068
    return 0.005

mapblklot_values['tax_rate'] = mapblklot_values['total_value'].apply(get_transfer_tax_rate)
mapblklot_values['adjusted_construction_cost'] = BASE_CONSTRUCTION_COST * (1 - mapblklot_values['tax_rate'] * TAX_TO_COST_MULTIPLIER)

print(f"\nTax bracket distribution:")
display(mapblklot_values['tax_rate'].value_counts().sort_index())


Tax bracket distribution:


tax_rate
0.0050       181
0.0068     28825
0.0075    125255
0.0225      5400
0.0550      2507
0.0600       442
Name: count, dtype: int64

In [5]:
with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f:
    mapblklot_values[['mapblklot', 'adjusted_construction_cost']].to_csv(f, index=False)
    costs_csv_path = f.name

print("Running housing projection model...")

result = subprocess.run(
    ['npx', 'vite-node', 'analyses/transfer-tax-reform/calculate-expected-units.mjs', costs_csv_path],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)

import os
os.unlink(costs_csv_path)

if result.returncode != 0:
    print("Error running calculation:")
    print("STDERR:", result.stderr)
    print("STDOUT:", result.stdout)
else:
    results = json.loads(result.stdout)
    print("Calculation complete!")

Running housing projection model...


Calculation complete!


In [6]:
results_df = pd.DataFrame({
    'Scenario': ['Low Growth', 'High Growth'],
    'Original (no reform)': [f"{results['original']['low']:,}", f"{results['original']['high']:,}"],
    'With Reform': [f"{results['withReform']['low']:,}", f"{results['withReform']['high']:,}"],
    'Difference': [
        f"+{results['difference']['low']:,} (+{results['difference']['lowPct']}%)",
        f"+{results['difference']['high']:,} (+{results['difference']['highPct']}%)"
    ]
})

display(results_df)

,Scenario,Original (no reform),With Reform,Difference
0,Low Growth,"44,918","49,776","+4,858 (+10.8%)"
1,High Growth,"76,411","85,372","+8,961 (+11.7%)"


## Results Summary

### Key Finding

Eliminating San Francisco's transfer tax would result in an estimated **+4,858 to +8,961 additional housing units** over 20 years, representing a **10.8-11.7% increase** over the baseline projection.

### Methodology Notes

- **Market values**: Loaded from `estimate-parcel-market-value/output.csv` (gradient boosting model trained on recent sales data)
- **Transfer tax impact**: Land value is ~2x construction costs in SF. Eliminating the transfer tax reduces land acquisition costs, with a 2x effect on effective construction cost index.
- **Model**: Uses the City Economist's probability/units model for housing projection.
- **Tax brackets**: SF transfer tax rates range from 0.5% to 6% based on property value.

### SF Transfer Tax Brackets

| Property Value | Tax Rate |
|----------------|----------|
| $100 - $250,000 | 0.50% |
| $250,001 - $999,999 | 0.68% |
| $1,000,000 - $4,999,999 | 0.75% |
| $5,000,000 - $9,999,999 | 2.25% |
| $10,000,000 - $24,999,999 | 5.50% |
| $25,000,000+ | 6.00% |

### Data Sources

- Market values from `estimate-parcel-market-value/output.csv`
- City Economist housing projection model

---
*Analysis performed: March 2025*